In [ ]:
from datasets import load_dataset

# FEVER

In [2]:
fever = load_dataset("fever", 'v1.0')

In [ ]:
fever['train'][2]

In [4]:
wiki_pages = load_dataset("fever", "wiki_pages")

In [ ]:
wiki_pages['wikipedia_pages'][:3]

## Match evidence from wiki_pages to FEVER

In [6]:
from tqdm import tqdm
import pandas as pd

In [ ]:
wiki_lookup = {}
for item in wiki_pages['wikipedia_pages']:
    article_id = item['id']

    if not article_id:
        continue

    sentences = {}

    for l in item['lines'].split('\n'):
        parts = l.split('\t')
        # at least 2 parts: sentence_id and sentence
        if len(parts) >= 2:
            try:
                sentence_id = int(parts[0])
                sentence_text = parts[1]

                sentences[sentence_id] = sentence_text

            except ValueError:
                continue

    wiki_lookup[article_id] = sentences

final_fever = []
dropped_count = 0

print("Finished with wiki")

for example in tqdm(fever['train'], total=len(fever["train"])):
    claim_id = example['id']
    claim = example['claim']
    label = example['label']

    evidence_article = example['evidence_wiki_url']
    evidence_sentence_id = example['evidence_sentence_id']

    evidence_text = None
    evidence_url = None


    if label != 'NOT ENOUGH INFO':
        if evidence_article and evidence_sentence_id is not None:
            article = wiki_lookup.get(evidence_article)
            if article and evidence_sentence_id in article:
                evidence_text = evidence_text = article[evidence_sentence_id]
                evidence_url = f"https://en.wikipedia.org/wiki/{evidence_article}"

            else:
                # drop entry if evidence not found
                dropped_count += 1
                continue

        else:
            dropped_count += 1
            continue      

    else:
        evidence_url = None
        evidence_text = ""      


    verifiable = "NOT VERIFIABLE" if label == 'NOT ENOUGH INFO' else "VERIFIABLE"
    human_verified = "NO"


    final_fever.append({
        "id": claim_id,
        "claim": claim,
        "label": label,
        "claimURL": None,
        "evidence_url": evidence_url,
        "evidence_text": evidence_text,
        "human_verified": human_verified,
        "verifiable": verifiable,
    })

df_final = pd.DataFrame(final_fever)

In [ ]:
import gc
del wiki_pages
del fever
del final_fever
gc.collect()

In [ ]:
dropped_count

In [ ]:
df_final.shape[0]

In [ ]:
df_final.head(10)

In [ ]:
print("Is 'id' unique?", df_final['id'].is_unique)

duplicates = df_final[df_final.duplicated(subset=['id'], keep=False)]
if not duplicates.empty:
    print("Duplicate IDs found:")
    print(duplicates)
else:
    print("No duplicate IDs found.")


duplicates_count = df_final['id'].duplicated().sum()
print("Number of duplicate IDs:", duplicates_count)

In [ ]:
# group by id - some claims have multiple evidence
aggregated = df_final.groupby('id', as_index=False).agg({
    "claim": "first",
    "label": "first",
    "claimURL": "first",
    "human_verified": "first",
    "verifiable": "first",
    "evidence_url": lambda urls: list(urls),
    "evidence_text": lambda texts: list(texts)
})

print("Number of unique claims after unification:", len(aggregated))

In [ ]:
for idx, row in aggregated.iterrows():
    print(f"Claim ID: {row['id']}")
    print(f"Claim: {row['claim']}")
    print(f"Label: {row['label']}")
    print(f"Claim URL: {row['claimURL']}")
    print(f"Human Verified: {row['human_verified']}")
    print(f"Verifiable: {row['verifiable']}")
    print("Evidence:")

    for url, text in zip(row['evidence_url'], row['evidence_text']):
        print(f"  - URL: {url}")
        print(f"    Text: {text}")
    print("-" * 80)

In [ ]:
# reassign claim ids from 1 to n to avoid clashes with multifc
df_result = aggregated.copy()  
df_result['id'] = range(1, len(aggregated) + 1)

print(df_result.head())
print("Total number of entries:", len(df_result))

In [ ]:
for idx, row in df_result.iterrows():
    print(f"Claim ID: {row['id']}")
    print(f"Claim: {row['claim']}")
    print(f"Label: {row['label']}")
    print(f"Claim URL: {row['claimURL']}")
    print(f"Human Verified: {row['human_verified']}")
    print(f"Verifiable: {row['verifiable']}")
    print("Evidence:")

    for url, text in zip(row['evidence_url'], row['evidence_text']):
        print(f"  - URL: {url}")
        print(f"    Text: {text}")
    print("-" * 80)

In [ ]:
df_result.shape[0]

# MultiFC